# Board Risk Report — Monthly Dashboard

> **Internal governance document** produced by the Risk Function under **AIFMD Art. 15**. Presented to the Risk Committee and Board of the AIFM within 5 business days of month end.
>
> **This is NOT the Annex IV regulatory submission** to the CSSF (which is the prescribed filing under AIFMD Art. 110). **This is NOT an investor report.** It is the internal evidence that the Board is actively receiving and reviewing risk information — the document the CSSF inspects under Art. 15.

---

## Scope

| Fund | Type | Strategy |
|---|---|---|
| AIFM Hedge Fund | AIFM | Long/Short Equity & Credit |
| AIFM Private Debt | AIFM | Senior Secured Lending |
| AIFM Real Estate | AIFM | Direct Property |
| UCITS Balanced | UCITS | Multi-Asset Balanced |

## Report sections

1. **Executive Summary** — fund overview, NAV, returns, RAG traffic lights
2. **VaR & Risk Metrics** — rolling 60-day VaR, limit utilisation
3. **Stress Test Results** — Annex VI severity heatmap, worst-case per fund
4. **Liquidity Monitoring** — ESMA bucket analysis, investor concentration
5. **Limit Breach Log** — all events, root cause, management action, status


In [ ]:
import warnings
warnings.filterwarnings("ignore")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Image
import io

from src.data.database import get_engine
from src.reporting.board_report import (
    generate_board_report,
    FUND_CONFIG, _LIQUID_FUNDS,
    _load_fund_metrics,
    _page_executive, _page_var, _page_stress,
    _page_liquidity, _page_breach_log,
    C,
)

ENGINE         = get_engine()
VALUATION_DATE = "2026-05-13"
OUTPUT_DIR     = "data"

## Load fund metrics

In [ ]:
# Load all fund metrics (VaR, returns, liquidity, stress, RAG status)
print("Loading fund metrics...")
all_metrics = {}
for fid in _LIQUID_FUNDS:
    print(f"  {fid}...", end=" ", flush=True)
    m = _load_fund_metrics(ENGINE, fid, VALUATION_DATE)
    all_metrics[fid] = m
    print(f"NAV EUR {m['nav']/1e6:,.0f}M  VaR1d {m['var_1d']*100:.2f}%  RAG={m['rag']}")
print("Done.")


## Page 1 — Executive Summary

In [ ]:
fig1 = _page_executive(all_metrics, VALUATION_DATE)
buf = io.BytesIO()
fig1.savefig(buf, format="png", dpi=130, bbox_inches="tight",
             facecolor=C["bg"])
buf.seek(0)
with open("../images/board_report_p1.png", "wb") as f:
    f.write(buf.read())
    
plt.close(fig1)
buf.seek(0)
display(Image(buf.read()))


## Page 2 — VaR & Risk Metrics

In [ ]:
fig2 = _page_var(all_metrics, VALUATION_DATE)
buf = io.BytesIO()
fig2.savefig(buf, format="png", dpi=130, bbox_inches="tight",
             facecolor=C["bg"])
plt.close(fig2)
buf.seek(0)
display(Image(buf.read()))


## Page 3 — Stress Test Results

In [ ]:
fig3 = _page_stress(all_metrics, VALUATION_DATE)
buf = io.BytesIO()
fig3.savefig(buf, format="png", dpi=130, bbox_inches="tight",
             facecolor=C["bg"])
plt.close(fig3)
buf.seek(0)
display(Image(buf.read()))


## Page 4 — Liquidity Monitoring

In [ ]:
fig4 = _page_liquidity(all_metrics, VALUATION_DATE)
buf = io.BytesIO()
fig4.savefig(buf, format="png", dpi=130, bbox_inches="tight",
             facecolor=C["bg"])

buf.seek(0)
with open("../images/board_report_p4.png", "wb") as f:
    f.write(buf.read())

plt.close(fig4)
buf.seek(0)
display(Image(buf.read()))


## Page 5 — Limit Breach Log

In [ ]:
fig5 = _page_breach_log(all_metrics, VALUATION_DATE)
buf = io.BytesIO()
fig5.savefig(buf, format="png", dpi=130, bbox_inches="tight",
             facecolor=C["bg"])
plt.close(fig5)
buf.seek(0)
display(Image(buf.read()))


## Export to PDF

In [ ]:
# Export all 5 pages to a single PDF for board distribution
pdf_path = generate_board_report(
    engine=ENGINE,
    valuation_date=VALUATION_DATE,
    output_dir=OUTPUT_DIR,
)
print(f"PDF written: {pdf_path}")
